In [1]:
import sys
print(sys.version)
print(spark.version)

3.8.5 | packaged by conda-forge | (default, Aug 29 2020, 01:22:49) 
[GCC 7.5.0]
3.0.1


In [2]:
import pandas as pd
import numpy as np
import json
pd.set_option('display.max_colwidth', None)
pd.reset_option('display.max_rows')
from itertools import compress 
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *
import seaborn as sns
import matplotlib.pyplot as plt
import re
warnings.filterwarnings(action='ignore')
import time
from datetime import datetime
import requests

import pandas as pd
from pandas.io.json import json_normalize


In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled",True)

In [4]:
#!pip install elasticsearch
#!pip install elasticsearch_dsl

In [5]:

#from elasticsearch import Elasticsearch, helpers
#import certifi

#from elasticsearch_dsl import connections, Search, Q
#from elasticsearch_dsl.query import MultiMatch, Match

## Read the tweets

In [6]:
!hadoop fs -du -s -h 'gs://msca-bdp-tweets/Tweets/'

2.3 T  2.3 T  gs://msca-bdp-tweets/Tweets


In [5]:
directory = 'gs://msca-bdp-tweets/Tweets/'
file = '*.json'
path = directory + file

In [ ]:
%time tweets_df = spark.read.json(path)

CPU times: user 240 ms, sys: 105 ms, total: 344 ms
Wall time: 26min 21s


In [ ]:
# check tweets count
tweets_df.count()

349730081

In [ ]:
# cache tweets df
tweets_df.limit(100000).cache().first()

contributors,coordinates,created_at,display_text_range,entities,extended_entities,extended_tweet,favorite_count,favorited,filter_level,geo,id,id_str,in_reply_to_screen_name,in_reply_to_status_id,in_reply_to_status_id_str,in_reply_to_user_id,in_reply_to_user_id_str,is_quote_status,lang,limit,place,possibly_sensitive,quote_count,quoted_status,quoted_status_id,quoted_status_id_str,quoted_status_permalink,reply_count,retweet_count,retweeted,retweeted_status,scopes,source,text,timestamp_ms,truncated,user,withheld_copyright,withheld_in_countries
null,null,Tue Sep 26 14:15:...,null,"[[[[0, 10], HireT...",null,null,0,false,low,null,912681921251020800,912681921251020800,null,null,null,null,null,false,en,null,null,false,0,null,null,null,null,0,0,false,null,null,"<a href=""http://b...",#HireTrans Chicag...,1506435301493,false,"[false, Fri May 0...",null,null
null,null,Tue Sep 26 14:15:...,null,"[[[[0, 11], NowPl...",null,null,0,false,low,null,912681922467418118,912681922467418118,null,null,null,null,null,false,en,null,null,false,0,null,null,null,null,0,0,false,null,null,"<a href=""https://...",#NowPlaying Chica...,1506435301783,false,"[false, Thu Jul 0...",null,null
null,null,Tue Sep 26 14:15:...,null,"[[],, [], [[twitt...",null,null,0,false,low,null,912681923859841027,912681923859841027,null,null,null,null,null,true,en,null,null,false,0,"[,, Tue Sep 26 13...",912676795081789440,912676795081789440,null,0,0,false,"[,, Tue Sep 26 13...",null,"<a href=""http://t...",RT @EWErickson: H...,1506435302115,false,"[false, Tue Feb 0...",null,null
null,null,Tue Sep 26 14:15:...,null,"[[],, [], [], [[7...",null,null,0,false,low,null,912681929220280320,912681929220280320,null,null,null,null,null,false,en,null,null,null,0,null,null,null,null,0,0,false,"[,, Tue Sep 26 13...",null,"<a href=""http://t...",RT @ShaunKing: Th...,1506435303393,false,"[false, Fri Apr 1...",null,null
null,null,Tue Sep 26 14:15:...,"[0, 69]","[[[[36, 45], foun...","[[[,, pic.twitter...",null,0,false,low,null,912681928368820227,912681928368820227,null,null,null,null,null,false,en,null,null,false,0,null,null,null,null,0,0,false,null,null,"<a href=""http://w...",Found dog in Chic...,1506435303190,false,"[false, Mon Feb 2...",null,null
null,null,Tue Sep 26 14:15:...,"[14, 47]","[[],, [], [], [[9...",null,null,0,false,low,null,912681929673031680,912681929673031680,jeffborzello,912675077660934145,912675077660934145,96845237,96845237,false,en,null,null,null,0,null,null,null,null,0,0,false,null,null,"<a href=""http://t...",@jeffborzello Uni...,1506435303501,false,"[false, Fri Dec 1...",null,null
null,null,Tue Sep 26 14:15:...,null,"[[],, [], [], [[2...",null,null,0,false,low,null,912681930033942529,912681930033942529,null,null,null,null,null,false,en,null,null,null,0,null,null,null,null,0,0,false,"[,, Tue Sep 26 13...",null,"<a href=""http://t...",RT @JasonRileyWDR...,1506435303587,false,"[false, Fri Mar 0...",null,null
null,null,Tue Sep 26 14:15:...,null,"[[],, [], [], [[1...",null,null,0,false,low,null,912681932269527042,912681932269527042,null,null,null,null,null,false,en,null,null,null,0,null,null,null,null,0,0,false,"[,, Mon Sep 25 20...",null,"<a href=""http://t...",RT @Elevate_Energ...,1506435304120,false,"[false, Mon Mar 0...",null,null
null,null,Tue Sep 26 14:15:...,null,"[[],, [], [[fb.me...",null,null,0,false,low,null,912681931807924224,912681931807924224,null,null,null,null,null,false,en,null,null,false,0,null,null,null,null,0,0,false,null,null,"<a href=""http://w...",Early Modern Rome...,1506435304010,false,"[false, Wed Mar 0...",null,null
null,null,Tue Sep 26 14:15:...,null,"[[],, [], [], [[9...",null,null,0,false,low,null,912681933691383808,912681933691383808,null,null,null,null,null,false,en,null,null,null,0,null,null,null,null,0,0,false,"[,, Tue Sep 26 14...",null,"<a href=""http://t...",RT @Dwyer_Coogan:...,1506435304459,false,"[false, Mon Jan 2...",null,null


In [ ]:
# tweets_df.printSchema()

## 1) Identify tweets related to UChicago and 3-4 universities of your choice
 

In [8]:

univ_tweets_df =tweets_df.filter((tweets_df['text'].contains('UChicago')))

In [ ]:
%time univ_tweets_df.count()

320312

In [ ]:
tweets_df.limit(10000).cache().show(5)

In [12]:
#user features
%time tweets_data=tweets_df.select(['created_at','reply_count','retweet_count','text','user.followers_count',
                              'user.id','user.friends_count','user.location', 'user.name'])

In [13]:
%time tweets_df_uchicago=tweets_data.filter(tweets_data['text'].contains('UChicago' ))

In [14]:
%time tweets_df_uic=tweets_data.filter(tweets_data['text'].contains('UIC'))

In [15]:
%time tweets_df_Yale=tweets_data.filter(tweets_data['text'].contains('Yale'))

In [16]:
%time tweets_df_OhioState=tweets_data.filter(tweets_data['text'].contains('OhioStae'))

## 2) Discard irrelevant tweets


In [10]:
%time tweets_df_uc_clean=tweets_df_uchicago.filter(col('text').isNotNull()).cache()

In [ ]:
%time tweets_df_uic_clean=tweets_df_uic.filter(col('text').isNotNull()).cache()

In [ ]:
%time tweets_df_yale_clean=tweets_df_yale.filter(col('text').isNotNull()).cache()

In [ ]:
%time tweets_df_osu_clean=tweets_df_ohiostate.filter(col('text').isNotNull()).cache()

## 3) Complete thorough EDA to identify which variables you can use to profile the Twitterers

In [17]:
%time tweets_df_uchicago.printSchema()

root
 |-- created_at: string (nullable = true)
 |-- reply_count: long (nullable = true)
 |-- retweet_count: long (nullable = true)
 |-- text: string (nullable = true)
 |-- followers_count: long (nullable = true)
 |-- id: long (nullable = true)
 |-- friends_count: long (nullable = true)
 |-- location: string (nullable = true)
 |-- name: string (nullable = true)



In [22]:
%time tweets_df_uchicago = tweets_df_uchicago.withColumn("reply_count", tweets_df_uchicago["reply_count"].cast(IntegerType()))

In [23]:
%time tweets_df_uchicago = tweets_df_uchicago.withColumn("retweet_count", tweets_df_uchicago["retweet_count"].cast(IntegerType()))

In [24]:
%time tweets_df_uchicago = tweets_df_uchicago.withColumn("followers_count", tweets_df_uchicago["followers_count"].cast(IntegerType()))

In [25]:
%time tweets_df_uchicago = tweets_df_uchicago.withColumn("friends_count", tweets_df_uchicago["friends_count"].cast(IntegerType()))

In [ ]:
t%time weets_df_uchicago.describe(['retweet_count']).show()

In [ ]:
%time tweets_df_uchicago.describe(['followers_count']).show()

## 4) Identify the most prolific / influential Twitterers
- By message volume
- By message retweet
- How much are they tweeting about the Universities vs. other topics?

In [25]:
%time df_uchicago = tweets_df_uchicago.groupBy('id').agg(count('id').alias('count'))

In [ ]:
%time df_uchicago.orderBy('count',ascending=False).show(5)df_uchicago.orderBy('count',ascending=False).show(5)

+-------------------+-----+
|                 id|count|
+-------------------+-----+
|          131144285| 3333|
|           20270494| 2935|
|1114265593744588806| 2205|
|          417357386| 1965|
|         2229760152| 1524|
+-------------------+-----+
only showing top 5 rows



In [ ]:
#dfs=dfs.toPandas()

In [29]:
%time df_uchicago_retwt=tweets_df_uc_clean.groupBy('id').agg(sum('retweet_count').alias('sum_retwt_count'))

In [ ]:
%time df_uchicago_retwt.orderBy('sum_retwt_count',ascending=False).show(5)

In [ ]:
%time df_retweets={'university':['Uchicago','UIC','Yale','OhioState'],
   'mean_retweet_count':[tweets_df_uc_clean.agg(round(mean('retweet_count')).cast('interger')).toPandas().iloc[0].values[0],
                               tweets_df_uic_clean.agg(round(mean('retweet_count')).cast('interger')).toPandas().iloc[0].values[0],
                               tweets_df_yale_clean.agg(round(mean('retweet_count')).cast('interger')).toPandas().iloc[0].values[0],
                               tweets_df_osu_clean.agg(round(mean('retweet_count')).cast('interger')).toPandas().iloc[0].values[0]]
   }
tweeters_user_count =pd.DataFrame(data=df_retweets=)
tweeters_user_count

## 5) Where are these Twitterers located?
 - For UChicago
 - For other universities
 - Do you see any relationship between university locations and Twitterers’ locations?
 - Visualize the relationships

In [ ]:
%time teewts_df_uchicago_userloc=tweets_df_uc_clean.filter(col('location').isNotNull()).\
groupBy(['loction']).\
agg(count('id').alias('total_count_in_location')).\
orderBy('total_count_in_location',ascending=False).limit(5).toPandas()

In [ ]:
%time teewts_df_uic_userloc=tweets_df_uic_cleanfilter(col('location').isNotNull()).\
groupBy(['loction']).\
agg(count('id').alias('total_count_in_location')).\
orderBy('total_count_in_location',ascending=False).limit(5).toPandas()

In [ ]:
%time teewts_df_osu_userloc=tweets_df_osu_clean.filter(col('location').isNotNull()).\
groupBy(['loction']).\
agg(count('id').alias('total_count_in_location')).\
orderBy('total_count_in_location',ascending=False).limit(5).toPandas()

In [ ]:
%time teewts_df_yale_userloc=tweets_df_yale_clean.filter(col('location').isNotNull()).\
groupBy(['loction']).\
agg(count('id').alias('total_count_in_location')).\
orderBy('total_count_in_location',ascending=False).limit(5).toPandas()

In [ ]:
%time my_plot =teewts_df_uchicago_userloc.plot(kind='bar',x='location', y='total_count_in_location', legend=None, title="Height by Gender")
my_plot.set_xlabel("Gender")
my_plot.set_ylabel("mean_user_friend")

## 6) What distinguishes University of Chicago Twitterers vs Twitterers who tweet about other universities
 - Visualize the trends

In [ ]:
%time d1={'university':['Uchicago','UIC','Yale','OhioState'],
   'tweeters_user_count':[tweets_df_uc_clean.agg(round(mean('followers_count')).cast('interger')).toPandas().iloc[0].values[0],
                               tweets_df_uic_clean.agg(round(mean('followers_count')).cast('interger')).toPandas().iloc[0].values[0],
                               tweets_df_yale_clean.agg(round(mean('followers_count')).cast('interger')).toPandas().iloc[0].values[0],
                               tweets_df_osu_clean.agg(round(mean('followers_count')).cast('interger')).toPandas().iloc[0].values[0]]
   }
tweeters_user_count_loc =pd.DataFrame(data=d1)
tweeters_user_count_loc

In [ ]:
%time my_plot = tweeters_user_count_loc.plot(kind='bar',x='university', y='tweeters_user_count', legend=None, title=" Trends")
#my_plot.set_xlabel("Gender")
#my_plot.set_ylabel("Height")

In [ ]:
%time d2={'university':['Uchicago','UIC','Loyola','OhioState'],
   'mean_user_friends_count':[tweets_df_uc_clean.agg(round(mean('friends_count')).cast('interger')).toPandas().iloc[0].values[0],
                               tweets_df_uic_clean.agg(round(mean('friends_count')).cast('interger')).toPandas().iloc[0].values[0],
                               tweets_df_yale_clean.agg(round(mean('friends_count')).cast('interger')).toPandas().iloc[0].values[0],
                               tweets_df_osu_clean.agg(round(mean('friends_count')).cast('interger')).toPandas().iloc[0].values[0]]
   }
tweeters_user_count_friends =pd.DataFrame(data=d2)
tweeters_user_count_friends

In [ ]:
%time my_plot = tweeters_user_count_friends.plot(kind='bar',x='university', y='tweeters_user_count', legend=None, title=" Tre

## 7) What are the timelines of these tweets? Do you see significant peaks and valleys?
- Do you see data collection gaps?

In [ ]:
from pyspark.sql.types import TimestampType
from pyspark.sql.functions import unix_timestamp

%time tweets_df3=tweets_df_uc_clean.withColumn("Date", unix_timestamp('created_at','MM/dd/yyyy HH:mm:ss').cast('timestamp'))

In [ ]:
tweets_date_data.describe(['date']).show()

In [ ]:
from pyspark.sql.types import DateType
from pyspark.sql.functions import lit, month, year

%time df3 = tweets_df3.withColumn('Month', month(df2['Date']))


In [ ]:
from pyspark.sql.functions import date_format
from pyspark.sql.functions import month, year

df4 = df3.withColumn('weekday', date_format('Date','E'))                 

In [ ]:
df5= df4.withColumn('Year', year(df4['Date']))

In [ ]:
from pyspark.sql.functions import col, column

In [ ]:
df_time=df6.groupBy('Year','Month').count().orderBy(["Year","Month"],ascending=[0,0])  

In [ ]:
cfiveyr = df_time.filter(df_time['Year']>'2014')
pdf = cfiveyr.toPandas()
pdf.plot(y="count", figsize=(12,4),style ="")


## 8) How unique are the messages for each of these universities?
- Are they mostly unique? Or mostly people are just copy-pasting the same text?
- You can use something like Jaccard similarity / Cosine Similarity / Simhash / Minhash to measure uniqueness / similarity
- Visualize message duplication (for each university – not between the universities)
- Please note: this is not a topic modeling (LDA / LSA) – but text similarity analysis.

In [ ]:
import re
from pyspark.ml.feature import MinHashLSH
from pyspark.ml.feature import CountVectorizer,  IDF, CountVectorizerModel, Tokenizer, RegexTokenizer, StopWordsRemover
from pyspark import SparkContext
from pyspark.sql import SQLContext
from pyspark.sql import Row
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
# df_text_raw = tweets_df_uchicago.select(["text"]).withColumnRenamed('title', 'text')
df_text_raw = tweets_df_uchicago.select(["text"])

In [ ]:
df_text_raw.limit(5)

In [ ]:
# Step 1. Clean the data, remove stopwords and create index
text = df_text_raw.rdd.map(lambda x : x['text']).filter(lambda x: x is not None)

StopWords = stopwords.words("english")

tokens = text\
    .map( lambda document: document.strip().lower())\
    .map( lambda document: re.split(" ", document))\
    .map( lambda word: [x for x in word if len(x) > 1] )\
    .zipWithIndex()



In [ ]:
row = Row('text')
df_text = text.map(row).zipWithIndex().toDF(['text','id'])
df_text.limit(5)

In [ ]:
df_text.count()

In [ ]:
df_tokens = spark.createDataFrame(tokens, ["list_of_words",'id'])

#Drop records with no tokens
df_tokens = df_tokens.where(col('list_of_words').getItem(0).isNotNull())
df_tokens.limit(5).toPandas()

In [ ]:
df_tokens.printSchema()

In [ ]:
df_tokens.count()

In [ ]:
# Step 2. Fit countvectorizer to create word features
vectorize = CountVectorizer(inputCol="list_of_words", outputCol="features", minDF=1.0)
df_vectorize = vectorize.fit(df_tokens).transform(df_tokens)

In [ ]:
df_vectorize.limit(5).toPandas()

In [ ]:
#Step 3. Fit MinHashLSH to create hash table
mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=5)
model = mh.fit(df_vectorize)
df_hashed = mh.fit(df_vectorize).transform(df_vectorize).cache()

In [ ]:
df_hashed_text = df_text.join(df_hashed, "id", how = 'left').cache()
df_hashed_text.limit(5)

In [ ]:
#Step 4. Establish similarity threshold and return near-duplicate records
jaccard_distance = 0.001

df_dups_text = model.approxSimilarityJoin(df_hashed_text, df_hashed_text, jaccard_distance).filter("datasetA.id < datasetB.id").select(
            col("distCol"),
            col("datasetA.id").alias("id_A"),
            col("datasetB.id").alias("id_B"),
            col('datasetA.text').alias('text_A'),
            col('datasetB.text').alias('text_B'),
#             col('datasetA.list_of_words').alias('words_A'),
#             col('datasetB.list_of_words').alias('words_B')
            )

In [ ]:
df_dups_30 = df_dups_text
df_dups_text.cache()
df_dups_text.limit(5).toPandas()

In [ ]:
records = df_hashed_text.count()
dups = df_dups_text.select('id_A').distinct().count()
uniques = records - dups

print ('Total records: ', records)
print ('Duplicate titles based on {', jaccard_distance, '} jaccard distance: ', dups)
print ('Unique titles based on {', jaccard_distance, '} jaccard distance: ', jaccard_distance, ': ', uniques)

In [ ]:
dups_df = pd.DataFrame.from_dict({'near_dups': [dups], 'unique': [uniques]})

ax=dups_df.plot(kind = 'bar',y=['near_dups', 'unique'], fontsize=10, color=['C0', 'C1'], align='center', width=0.8, xlabel="Duplicates vs. Unique")
ax.set_title('News title duplication analysis', fontsize=15)
for p in ax.patches:
       ax.annotate(format(p.get_height(), '.1f'), 
                   (p.get_x() + p.get_width() / 2., p.get_height()/2), 
                   ha = 'center', va = 'center', 
                   xytext = (0, 9), 
                   textcoords = 'offset points') 